[![img/pythonista.png](img/pythonista.png)](https://www.pythonista.io)

# Introducción a *Dask*.

* La siguiente celda instala *Dask* con soporte para *DataFrames* en el entorno actual en caso de que no esté disponible.

In [ ]:
%pip install dask[dataframe]

* La siguiente celda importa `polars` con el alias convencional `pl`, `dask.dataframe` con el alias `dd`, junto con `pandas`, `numpy` y `os`.

**Nota:** Por convención, `dask.dataframe` se importa con el alias `dd`.

In [ ]:
import polars as pl
import dask.dataframe as dd
import pandas as pd
import numpy as np
import os

* La siguiente celda define y en su caso crea los directorios que se usarán para almacenar los datos y los resultados de este capítulo.

In [ ]:
sample_dir = 'data'
temp_dir   = 'data_tmp'
os.makedirs(sample_dir, exist_ok=True)
os.makedirs(temp_dir,   exist_ok=True)

## Contexto: Del análisis local al escalado distribuido.

En los capítulos anteriores se estudiaron **Polars** y **PyArrow**, que ofrecen capacidades de alto rendimiento para analizar datos en una sola máquina. Sin embargo, cuando los datos crecen más allá de lo que puede caber en la memoria disponible, se necesita una estrategia diferente.

### ¿Polars o Dask?

| Aspecto | **Polars** | **Dask** |
|---------|-----------|----------|
| **Tamaño de datos** | Hasta ~100 GB en memoria | >100 GB (terabytes) |
| **Ejecución** | Una máquina | Múltiples máquinas (clúster) |
| **Velocidad (single-machine)** | 3-10× más rápido que *Pandas* | Overhead distribuido |
| **Facilidad de uso** | *API* simple y moderna | Requiere configuración de clúster |
| **Casos de uso** | ML local, analítica exploratoria | Big Data, ETL distribuido |
| **Integración Cloud** | Experimental | Madura (*AWS*, *GCP*, *Azure*) |

**Regla práctica:**
- Usar **Polars** cuando los datos caben en memoria (máxima velocidad).
- Usar **Dask** cuando los datos superan la capacidad de una máquina.

[*Dask*](https://dask.org/) es una biblioteca general para cómputo paralelo que permite escalar operaciones a través de clústers. A diferencia de *Polars*, que está optimizado para una sola máquina, *Dask* está diseñado para distribuir el trabajo entre múltiples equipos.

*Dask* consta de:

* Un calendarizador de tareas dinámico (*dynamic task scheduler*).
* Una colección de bibliotecas optimizadas para *Big Data* con interfaces que extienden a *NumPy* y *Pandas*.

https://docs.dask.org/en/stable/

https://tutorial.dask.org/

<img src="img/arquitectura_dask.png" width=75%>

## Principales paquetes de *Dask*.

### Colecciones de datos.

| Paquete | Alias | Descripción |
| :--- | :--- | :--- |
| [`dask.array`](https://docs.dask.org/en/stable/array.html) | `da` | Arreglos paralelos equivalentes a *NumPy*. |
| [`dask.dataframe`](https://docs.dask.org/en/stable/dataframe.html) | `dd` | *DataFrames* distribuidos equivalentes a *Pandas*. |
| [`dask.bag`](https://docs.dask.org/en/stable/bag.html) | `db` | Colecciones de datos semi-estructurados. |

### Bibliotecas de paralelismo.

| Paquete | Descripción |
| :--- | :--- |
| [`dask.delayed`](https://docs.dask.org/en/stable/delayed.html) | Paraleliza funciones de *Python* de forma declarativa. |
| [`dask.futures`](https://docs.dask.org/en/stable/futures.html) | Implementación de `concurrent.futures` optimizada para clústers. |

## Evaluación perezosa (*lazy*) con `.compute()`.

Al igual que *Polars*, *Dask* utiliza **evaluación lazy**: las operaciones se definen sin ejecutarse hasta que se llama explícitamente a `.compute()`. La diferencia es que *Dask* distribuye esa ejecución entre los núcleos disponibles o entre los *workers* de un clúster.

| Operación | Comportamiento |
| :--- | :--- |
| `dd.read_csv(path)` | Registra la intención de leer; **no carga datos**. |
| `.compute()` | Ejecuta el grafo y devuelve el resultado como *Pandas DataFrame*. |
| `.persist()` | Ejecuta el grafo y retiene el resultado en memoria (*caché*). |

## *Dataset* ilustrativo.

Para garantizar la reproducibilidad del notebook se genera un *dataset* sintético con series epidemiológicas: 200 registros diarios con columnas de casos por entidad (`Nacional`, `CDMX`, `Jalisco`, `NuevoLeon`).

* La siguiente celda usa **Polars** para crear el *dataset* de ejemplo y guardarlo como CSV. Este es el patrón recomendado: generar y preprocesar con *Polars* (rápido, en memoria), luego leer con *Dask* cuando se requiere escalar.

In [ ]:
from datetime import date, timedelta

np.random.seed(42)
n = 200
start    = date(2020, 3, 1)
end      = start + timedelta(days=n - 1)
fechas   = pl.date_range(start, end, interval='1d', eager=True)
nacional = np.random.randint(5_000, 120_000, n)

df_covid = pl.DataFrame({
    'index':     list(range(n)),
    'Fecha':     fechas.dt.strftime('%Y-%m-%d'),
    'Nacional':  nacional.tolist(),
    'CDMX':      (nacional * np.random.uniform(0.20, 0.40, n)).astype(int).tolist(),
    'Jalisco':   (nacional * np.random.uniform(0.08, 0.15, n)).astype(int).tolist(),
    'NuevoLeon': (nacional * np.random.uniform(0.06, 0.12, n)).astype(int).tolist(),
})

url_covid_csv = 'data/data_covid.csv'
df_covid.write_csv(url_covid_csv)
print(f'Dataset: {df_covid.shape[0]} filas, {df_covid.shape[1]} columnas')
print(df_covid.head())

* La siguiente celda importa `dask.dataframe` e invoca `dd.read_csv()`. **Ningún dato se carga en este momento** — *Dask* registra la intención de leer el archivo y devuelve un `DataFrame` lazy.

In [ ]:
df = dd.read_csv(url_covid_csv)
print(f'Tipo: {type(df)}')
print(f'Particiones: {df.npartitions}')

* La siguiente celda muestra el `DataFrame` de *Dask* (modo *lazy*): solo se presentan los metadatos (esquema y número de particiones), sin cargar ningún dato en memoria.

In [ ]:
df

* La siguiente celda llama a `.compute()` para materializar **todo** el CSV en un *DataFrame* de *Pandas*.

In [ ]:
df.compute()

* La siguiente celda accede a la columna `'Nacional'` y verifica su tipo con `type()`. El resultado es una `dask.dataframe.Series` (aún *lazy*).

In [ ]:
type(df['Nacional'])

* La siguiente celda aplica `.compute()` solo a la columna `'Nacional'`, cargando únicamente esa serie en memoria en lugar de todo el *DataFrame*.

In [ ]:
df['Nacional'].compute()

* La siguiente celda encadena un filtro y una selección de columnas de forma *lazy*, sin ejecutar ningún cómputo. El grafo de tareas se construye pero no se evalúa.

In [ ]:
query = df.loc[df['Nacional'] > 50000].loc[:, ['index', 'Nacional']]
print(f'Tipo: {type(query)}')

* La siguiente celda ejecuta la cadena de operaciones *lazy* con `.compute()`, cargando en memoria únicamente las filas y columnas que cumplen el filtro.

In [ ]:
query.compute()

### Persistencia en memoria con `persist()`.

Con evaluación perezosa, cada llamada a `.compute()` **recalcula todo el grafo** desde el origen. Cuando el mismo `DataFrame` se reutiliza en múltiples operaciones, esto implica releer y reprocesar los datos innecesariamente.

`.persist()` ejecuta el grafo **una sola vez** y retiene los resultados en memoria, comportándose como un caché distribuido.

| Operación | Comportamiento |
| :--- | :--- |
| `.compute()` | Ejecuta el grafo y devuelve el resultado a *Python*. |
| `.persist()` | Ejecuta el grafo y retiene el resultado en los *workers*. |

https://docs.dask.org/en/stable/api.html#dask.dataframe.DataFrame.persist

* La siguiente celda ilustra la diferencia entre `.compute()` y `.persist()`: sin `.persist()` cada operación relee el CSV desde disco; con `.persist()` los datos se cargan una sola vez y las operaciones siguientes operan sobre la copia en memoria.

In [ ]:
df = dd.read_csv(url_covid_csv)

# Sin persist: cada operación relee y recalcula desde disco
resultado_a = df[df['Nacional'] > 50000].compute()
resultado_b = df['Nacional'].mean().compute()

# Con persist: los datos se cargan una sola vez en memoria
df_cache = df.persist()

# Las siguientes operaciones son más rápidas
resultado_a = df_cache[df_cache['Nacional'] > 50000].compute()
resultado_b = df_cache['Nacional'].mean().compute()

print(f'Media Nacional: {resultado_b:.2f}')
print('Resultados calculados con datos en caché.')

### Otras bibliotecas de paralelismo.

* [`dask.delayed`](https://docs.dask.org/en/stable/delayed.html): permite paralelizar funciones arbitrarias de *Python* decorándolas con `@dask.delayed`. Útil para pipelines personalizados que no encajan en el modelo de *DataFrame*.
* [`dask.futures`](https://docs.dask.org/en/stable/futures.html): implementación de `concurrent.futures` optimizada para clústers *Dask*. Útil cuando se necesita control fino sobre la ejecución distribuida.

## Procesamiento de datos particionados.

El **particionado *Hive-style*** organiza archivos *Parquet* en subdirectorios según el valor de una columna:

```
datos/
  region=norte/
    part.0.parquet
  region=sur/
    part.0.parquet
  region=este/
    part.0.parquet
```

*Dask* puede leer únicamente las particiones necesarias (*partition pruning*), evitando leer archivos irrelevantes. Esto reduce significativamente el I/O en grandes volúmenes de datos.

https://docs.dask.org/en/stable/dataframe-parquet.html

* La siguiente celda usa **Polars** para crear los datos de ejemplo y escribirlos en formato *Parquet* particionado por `region` (*Hive-style*), generando un subdirectorio por cada valor único de la columna.

In [ ]:
ventas_url = 'data_tmp/ventas_particionadas'

df_ventas = pl.DataFrame({
    'region':   ['norte', 'norte', 'sur', 'sur', 'este'],
    'producto': ['A', 'B', 'A', 'C', 'B'],
    'ventas':   [100, 200, 150, 300, 250],
})

df_ventas.write_parquet(
    ventas_url,
    partition_by='region',
)
print('Datos escritos con particionado por región:')
print(df_ventas)

* La siguiente celda lee solo la partición `region='norte'` usando `filters=[('region', '==', 'norte')]`. *Dask* no abre los archivos de las demás regiones (*partition pruning*).

In [ ]:
df_norte = dd.read_parquet(
    ventas_url,
    filters=[('region', '==', 'norte')]
)

print(f'Particiones leídas: {df_norte.npartitions}')
print(df_norte.compute())

## Despliegue de un clúster con `Dask.Distributed`.

*Dask* puede desplegarse en clústers mediante varios equipos *workers* gestionados por un *scheduler* central. Este modo distribuido es el que permite procesar volúmenes de datos a escala de *Big Data* y es la puerta de entrada a plataformas cloud como *AWS*, *GCP* y *Azure*.

https://distributed.dask.org/en/stable/

<img src="img/dask_cluster.png" width=45%>

Un clúster de *Dask* se compone de:

* **Scheduler**: coordina la ejecución del grafo de tareas.
* **Workers**: ejecutan las tareas individuales en paralelo.
* **Client**: interfaz de *Python* para enviar tareas al clúster.

Para uso local, `dask.distributed.LocalCluster` simula un clúster completo en la misma máquina. En producción, existen integraciones nativas con *Kubernetes*, *YARN*, *AWS Fargate*, *Google Cloud Dataproc* y *Azure ML*.

## Integración: *Polars* → *Dask*.

Un patrón muy común en análisis de datos a escala es:

1. **Explorar y preprocesar** datos con **Polars** en la máquina local (rápido, simple).
2. **Escalar** con **Dask** cuando el volumen supera la memoria disponible.

El formato *Parquet* actúa como puente eficiente: *Polars* lo escribe con compresión óptima y *Dask* lo lee en paralelo con *partition pruning*, sin conversión intermedia.

```
Polars (exploración local)
       ↓  write_parquet()
    Parquet
       ↓  dd.read_parquet()
Dask (procesamiento distribuido)
       ↓  .compute()
    Resultado
```

* La siguiente celda ejemplifica el flujo completo *Polars* → *Parquet* → *Dask*: se generan datos con *Polars*, se guardan en *Parquet* con compresión `zstd` y se procesan con *Dask* aplicando un filtro y una agregación.

In [ ]:
from datetime import date, timedelta

# Paso 1: Procesar con Polars (rápido, single-machine)
start_fecha = date(2024, 1, 1)
df_origen = pl.DataFrame({
    'fecha':     [(start_fecha + timedelta(days=i)).isoformat() for i in range(1000)],
    'categoria': (['A', 'B', 'C', 'D'] * 250),
    'valor':     np.random.uniform(0, 100, 1000).tolist(),
})

url_integracion = 'data_tmp/datos_integracion.parquet'
df_origen.write_parquet(url_integracion, compression='zstd')
print(f'Polars: {df_origen.shape[0]} filas guardadas en Parquet (zstd)')

# Paso 2: Distribuir con Dask cuando sea necesario
df_dask = dd.read_parquet(url_integracion)
resultado = (
    df_dask
    .groupby('categoria')['valor']
    .mean()
    .compute()
    .sort_values()
)

print('\nMedia por categoría (procesado con Dask):')
print(resultado)

## Resumen.

*Dask* complementa a *Polars* como capa de escalado distribuido, utilizando el mismo paradigma de evaluación lazy y el formato *Parquet* como estándar de interoperabilidad:

| Concepto | Descripción |
| :--- | :--- |
| **`dd.read_csv()` / `dd.read_parquet()`** | Lectura *lazy*: registra la intención sin cargar datos. |
| **`.compute()`** | Materializa el grafo y devuelve un *DataFrame* de *Pandas*. |
| **`.persist()`** | Cachea el resultado en memoria para reutilización eficiente. |
| **Particionado *Hive-style*** | Organización de *Parquet* por columna para *partition pruning*. |
| **`dask.distributed`** | Clúster distribuido; puerta de entrada a *Cloud* y *Big Data*. |
| **Flujo recomendado** | Polars (local) → Parquet → Dask (distribuido). |

<p style="text-align: center"><a rel="license" href="http://creativecommons.org/licenses/by/4.0/"><img alt="Licencia Creative Commons" style="border-width:0" src="https://i.creativecommons.org/l/by/4.0/80x15.png" /></a><br />Esta obra está bajo una <a rel="license" href="http://creativecommons.org/licenses/by/4.0/">Licencia Creative Commons Atribución 4.0 Internacional</a>.</p>
<p style="text-align: center">&copy; José Luis Chiquete Valdivieso. 2017-2026.</p>